# 01. 데이터 품질 진단**이 노트북의 목적**: 분석을 시작하기 전에 *"이 데이터로 무엇을 말할 수 있고 무엇을 말할 수 없는가"* 를 먼저 확정합니다.여기서 나온 숫자는 그대로 `docs/FINDINGS.md` 의 "데이터 범위"와 대시보드 '데이터 신뢰도' 탭에 들어갑니다.> 시작 전에 `data/raw/` 에 CSV 8개가 있어야 합니다. 없으면 `data/README.md` 를 보세요.

In [ ]:
import duckdb, pandas as pdimport matplotlib, matplotlib.pyplot as pltpd.set_option("display.max_columns", 50)# 한글 깨짐 방지for _f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR", "Noto Sans KR"]:    if _f in {f.name for f in matplotlib.font_manager.fontManager.ttflist}:        plt.rcParams["font.family"] = _f        breakplt.rcParams["axes.unicode_minus"] = False# DuckDB는 CSV를 그대로 읽습니다. 적재 과정이 따로 없습니다.con = duckdb.connect()con.execute(open("../sql/base.sql", encoding="utf-8").read().format(data="../data/raw"))def q(sql):    """SQL을 실행해 pandas DataFrame으로 돌려줍니다."""    return con.sql(sql).df()q("SELECT count(*) AS 주문수 FROM order_fact")

## 0. 먼저 눈으로 보기`sql/base.sql` 이 만들어 둔 `order_fact` 는 **1행 = 주문 1건** 입니다.앞으로 모든 분석은 이 뷰 하나만 봅니다.

In [ ]:
q("DESCRIBE order_fact")

In [ ]:
q("SELECT * FROM order_fact LIMIT 5")

## 1. 이 데이터로 답할 수 있는 범위전체 주문 중 실제로 분석에 쓸 수 있는 건 몇 건인지 셉니다.`COUNT(*) FILTER (WHERE 조건)` 은 조건을 만족하는 행만 세는 문법입니다.

In [ ]:
q("""SELECT '전체 주문' AS 항목, count(*) AS 건수 FROM order_factUNION ALL SELECT '배송 완료',   count(*) FILTER (WHERE order_status = 'delivered') FROM order_factUNION ALL SELECT '수령일 있음',  count(*) FILTER (WHERE delivered_at IS NOT NULL) FROM order_factUNION ALL SELECT '리뷰 있음',   count(*) FILTER (WHERE has_review) FROM order_factUNION ALL SELECT '분석 가능 (지연일+리뷰)', count(*) FILTER (WHERE delay_days IS NOT NULL AND has_review) FROM order_factORDER BY 건수 DESC""")

**확인할 것** — '분석 가능' 건수가 전체의 몇 %인가요?이 비율이 낮다면 앞으로의 결론은 *전체 주문* 이 아니라 *그 부분집합* 에 대한 이야기입니다.그 사실을 결론에 반드시 함께 써야 합니다.

## 2. 결합 단계마다 표본이 얼마나 줄어드나테이블을 붙일 때마다 몇 건이 사라지는지 추적합니다.크게 줄어드는 단계가 곧 결론의 한계입니다.`JOIN` (= `INNER JOIN`) 은 양쪽에 다 있는 행만 남기고, `LEFT JOIN` 은 왼쪽을 전부 남깁니다.아래는 일부러 `JOIN` 을 써서 손실을 드러냅니다.

In [ ]:
q("""SELECT '① 원본 주문'        AS 단계, count(*) AS 건수 FROM ordersUNION ALL SELECT '② + 상품 결합', count(*) FROM orders JOIN order_items USING (order_id)UNION ALL SELECT '③ + 리뷰 결합', count(*) FROM orders JOIN order_items USING (order_id) JOIN reviews USING (order_id)UNION ALL SELECT '④ + 수령일 존재', count(*) FROM order_fact WHERE delay_days IS NOT NULL AND has_review""")

## 3. 중복 리뷰한 주문에 리뷰가 여러 건 달린 경우가 있습니다.점수가 서로 다르다면, **어느 것을 남길지가 결과를 바꾸는 결정** 이 됩니다.`sql/base.sql` 은 "가장 나중에 쓴 1건"을 남기도록 해 두었습니다. 그 선택이 타당한지 여기서 확인하세요.

In [ ]:
q("""SELECT count(*) AS 중복_주문수,       count(*) FILTER (WHERE mx <> mn) AS 점수가_다른_주문수,       round(avg(mx - mn), 2) AS 평균_점수차FROM (    SELECT order_id,           max(CAST(review_score AS INT)) AS mx,           min(CAST(review_score AS INT)) AS mn    FROM reviews_raw    WHERE review_score IS NOT NULL    GROUP BY order_id    HAVING count(*) > 1)""")

## 4. 【직접 작성】 재구매 고객은 몇 명인가**함정이 하나 있습니다.**- `customer_id` — 주문할 때마다 **새로 발급되는 값**. 사람이 아닙니다.- `customer_unique_id` — **실제 사람**.`customer_id` 로 세면 재구매율이 0%로 나옵니다. 실무에서 실제로 자주 나는 사고입니다.**할 일**: 아래 두 숫자를 각각 구하고, 차이가 왜 나는지 `docs/FINDINGS.md` 에 적으세요.1. `customer_id` 기준 "2회 이상 주문한 고객" 비율2. `customer_unique_id` 기준 "2회 이상 주문한 고객" 비율**힌트**```sqlWITH per_person AS (    SELECT <기준컬럼>, count(*) AS n    FROM order_fact    GROUP BY 1)SELECT count(*)                                AS 고객수,       count(*) FILTER (WHERE n >= 2)          AS 재구매고객수,       round(100.0 * count(*) FILTER (WHERE n >= 2) / count(*), 2) AS 재구매율FROM per_person````order_fact` 에는 `customer_unique_id` 만 있습니다. `customer_id` 기준은 `orders` 뷰를 쓰세요.

In [ ]:
# 여기에 작성하세요# q("""# ...# """)

## 정리아래 빈칸을 채워 `docs/FINDINGS.md` 의 "데이터 범위"로 옮기세요.- 전체 주문 **[   ]** 건 중 분석 가능 **[   ]** 건 (**[  ]**%)- 빠진 이유: 리뷰 없음 **[   ]** 건 / 수령일 결측 **[   ]** 건 / 취소·미배송 **[   ]** 건- 중복 리뷰 **[   ]** 건, 그중 점수가 다른 것 **[   ]** 건 → 최신 1건 채택- 재구매 고객 **[   ]** 명 (**[  ]**%) → 04 노트북에서 이 숫자가 결정적입니다- 따라서 이 분석의 결론은 **[ 어떤 부분집합 ]** 에 대한 이야기입니다